In this guide, we will be exploring the concept of agents and their various applications in AI (this content is derived from Huggingface Agents course)

What is an Agent? In simple definition it is an AI model capable of reasoning, planning, and interacting with its environment.

The following are some research papers to read:
- https://arxiv.org/pdf/2401.03428
- https://arxiv.org/pdf/2503.21460


An Agent has agency aka the ability to interact with the environment

For example, when receiving the user request, it undergoes the following:
- Step 1: Think and Plan
- Step 2: Act using Tools

Here is a more precise definition:
"An Agent is a system that leverages an AI model to interact with its environment in order to achieve a user-defined objective. It combines reasoning, planning, and the execution of actions (often via external tools) to fulfill tasks."

Think of the Agent as having 2 main parts:

1. The Brain (AI model)

  This is where all the thinking happens. The AI model handles reasoning and planning. It decides which Actions to take based on the situation


2. The Body

  This part represents everything the Agent is equipped to do. The scope of possible actions depends on what the agent has been equipped with.




The most common AI model found in Agents is an LLM (Large Language Model), which takes Text as an input and outputs Text as well.

Well known examples are GPT-4 from OpenAI, Llama from Meta, Gemini from Google, etc. These models have been trained on a vast amount of text and able to generalize well.

LLMs are amazing models but they can only generate text. However, if you ask a well-known chat application like HuggingChat or ChatGPT to generate an image, they can. How is this possible? The answer is that the developers of HuggingChat, ChatGPT and similar apps implemented additional functionality called Tools which the LLM can use to create images.



An Agent can perform any task we implement via Tools to complete Actions. The design of the Tools is very important and has a great impact on the quality of your Agent.
For example, if I write an Agent to act as my personal assistant on my computer, and ask it to "send an email to my Manager asking to delay today's meeting", I can give it some code to send emails.

In [ ]:
# The send_message tool defined as a function
def send_message_to(recipient, message):
    """Useful to send an e-mail message to a recipient"""
    ...

# LLM uses the send message tool/function
send_message_to("Manager", "Can we postpone today's meeting?")


To summarize, an Agent is a system that uses an AI model (usually an LLM) as its core reasoning engine, to:

- Understand natural language: Interpret and respond to human instructions in a meaningful way
- Reason and plan: Analyze information, make decisions, and devise strategies to solve problems
- Interact with its environment: Gather information, take actions, and observe the results of those actions

Let's now look at how LLM structure their generations through chat templates. When you chat with systems like ChatGPT or HuggingChat, you're actually exchanging messages. Behind the scenes, these messages are concatenated and formatted into a prompt that the model can understand. This is where chat templates come in. They act as the bridge between conversational messages and the specific formating requirements of your chosen LLM.

Next, lets look at System messages. System messages/prompts define how the model should behave. They serve as persistent instructions, guiding every subsequent interaction. For example:

In [ ]:
system_message = {
    "role": "system",
    "content": "You are a professional customer service agent. Always be polite, clear, and helpful."
}

With this system message, the Agent will respond in a polite and helpful manner.

When using Agents, the system message also gives information about the available tools, provides instructions to the model on how to format the actions to take, and includes guidelines on how the thought process should be segmented.

A conversation consists of alternating messages between a Human (user) and an LLM (assistant). Chat templates help maintain context by preserving conversation history, storing previous exchanges between the user and the assistant. This leads to more coherent multi-turn conversations.

In [ ]:
conversation = [
    {"role": "user", "content": "I need help with my order"},
    {"role": "assistant", "content": "I'd be happy to help. Could you provide your order number?"},
    {"role": "user", "content": "It's ORDER-123"},
]

In the above example, the user asks a question, the LLM asks for clarification and user responds. We always concatenate all the messages in the conversation and pass it to the LLM as a single stand-alone sequence. The chat template will convert all the messages inside this Python list into a prompt, which is just a string input that contains all the messages.

For the example below, this is how the SmoLM2 chat template would format the previous exchange into a prompt.

In [ ]:
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
I need help with my order<|im_end|>
<|im_start|>assistant
I'd be happy to help. Could you provide your order number?<|im_end|>
<|im_start|>user
It's ORDER-123<|im_end|>
<|im_start|>assistant

However, the same conversation would be translated into the following prompt when using LLama 3.2:

In [ ]:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 10 Feb 2025

<|eot_id|><|start_header_id|>user<|end_header_id|>

I need help with my order<|eot_id|><|start_header_id|>assistant<|end_header_id|>

I'd be happy to help. Could you provide your order number?<|eot_id|><|start_header_id|>user<|end_header_id|>

It's ORDER-123<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Chat templates are important for structuring conversations between language models and users. They guide how message exchanges are formatted into a single prompt.
Another important thing is we need to understand the difference between a Base Model vs. an Instruct Model:
- Base Model: trained on raw text data to predict the next token
- Instruct Model: fine-tuned specifically to follow instructions and engage in conversations.

In order to make a Base Model behave like an instruct model, we need to format our prompt in a consistent way that the model can understand. So this is where chat templates come in.



The `transformers` library will take care of chat templates as part of the tokenization process. All we have to do is structure our messages in the correct way and the tokenizer will take care of the rest.

In [ ]:
messages = [
    {"role": "system", "content": "You are an AI assistant with access to various tools."},
    {"role": "user", "content": "Hi !"},
    {"role": "assistant", "content": "Hi human, what can help you with ?"},
]

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-1.7B-Instruct")
rendered_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

Another important aspect of AI Agents is their ability to take actions. This can be done through the use of Tools.

A Tool is a function given to the LLM. This function should fulfill a clear objective. Here are some commonly used tools in AI agents.

- Web Search = Allows the agent to fetch up-to-date info from the Internet
- Image Generation = Creates images based on text descriptions
- Retrieval = Retrieves information from an external source
- API Interface = Interacts with an external API (GitHub, YouTube, Spotify, etc)

A good tool should be something that complements the power of an LLM. For example, if you need to perform arithmetic, giving a calculator tool to your LLM will provide better results than relying on the native capabilities of the model.

A Tool should contain:
- A textual description of what the function dows
- A Callable (something to perform an action)
- Arguments with typings
- (Optional) Outputs with typings

Here is an example using Python to implement Tools

In [ ]:
@tool
def calculator(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

print(calculator.to_string())

We can also create a Generic Tool class to reuse whenever we need to use a tool

In [ ]:
from typing import Callable


class Tool:
    """
    A class representing a reusable piece of code (Tool).

    Attributes:
        name (str): Name of the tool.
        description (str): A textual description of what the tool does.
        func (callable): The function this tool wraps.
        arguments (list): A list of arguments.
        outputs (str or list): The return type(s) of the wrapped function.
    """
    def __init__(self,
                 name: str,
                 description: str,
                 func: Callable,
                 arguments: list,
                 outputs: str):
        self.name = name
        self.description = description
        self.func = func
        self.arguments = arguments
        self.outputs = outputs

    def to_string(self) -> str:
        """
        Return a string representation of the tool,
        including its name, description, arguments, and outputs.
        """
        args_str = ", ".join([
            f"{arg_name}: {arg_type}" for arg_name, arg_type in self.arguments
        ])

        return (
            f"Tool Name: {self.name},"
            f" Description: {self.description},"
            f" Arguments: {args_str},"
            f" Outputs: {self.outputs}"
        )

    def __call__(self, *args, **kwargs):
        """
        Invoke the underlying function (callable) with provided arguments.
        """
        return self.func(*args, **kwargs)

We can then create a Tool with this class using the following implementation

In [ ]:
calculator_tool = Tool(
    "calculator",                   # name
    "Multiply two integers.",       # description
    calculator,                     # function to call
    [("a", "int"), ("b", "int")],   # inputs (names and types)
    "int",                          # output
)

Then for our LLM we can do this:

In [ ]:
Tool Name: calculator, Description: Multiply two integers., Arguments: a: int, b: int, Outputs: int

Let's now look at the core components of an Agent.
- Thought: The LLM part of the Agent decides what the next step should be
- Action: The agent takes an action by calling the tools with the associated arguments
- Observation: The model reflects on the response from the tool

In many Agent frameworks, the rules and guidelines are embedded directly into the system prompts, ensuring that every cycle adheres to a defined logic. In a System Prompt, we defined:

- The Agent's behavior (You are an AI assistant designed to help users efficiently and accurately)
- The Tools our Agent has access to (You have access to the following tools:Tool Name: calculator, Description: Multiply two integers., Arguments: a: int, b: int, Outputs: int)
- The Thought-Action-Observation Cycle (You should first think step by step in order to fulfill the objective with a reasoning divided into Thought/Action/Observation steps that can be repeated multiple times if needed)


The Chain-of-Thought (CoT) is a prompting technique that guides a model to think through a problem step-by-step before producing a final answer. For example:

In [ ]:
Question: What is 15% of 200?
Thought: Let's think step by step. 10% of 200 is 20, and 5% of 200 is 10, so 15% is 30.
Answer: 30

ReAct is an approach that combines "Reasoning" (Think) with "Acting" (Act). This a prompting technique that encourages the model to think step-by-step and interleave actions (like using tools) between reasoning steps

In [ ]:
Thought: I need to find the latest weather in Paris.
Action: Search["weather in Paris"]
Observation: It's 18°C and cloudy.
Thought: Now that I know the weather...
Action: Finish["It's 18°C and cloudy in Paris."]

Both CoT and ReAct are both step-by-step logic. CoT doesn't require external tools while ReAct may require them. CoT is best suited for logic, math, internal tasks, while ReAct is best suited for info-seeking and dynamic multi-step tasks.